## Lab 3: Clean new data

Today we cleaned a messy table step by step: text, types, dates and duplicates. Cleaning is
something you learn by doing it, and the best test is a table you have not seen before. So in this
lab you get new data: the loans of a library. The library and its data are made up, but the
problems in the table are of the kinds that real data has. This lab has two parts, and you choose.

Part 1: Practise repeats the steps from today on the new table. The problems are the same kinds
as today, in new places. Do this part if you want to try the whole routine once more on your own.

Part 2: Go deeper is for you who felt "I understood this, and I want to go further". The questions
are more open, and you have to decide what is right when the data is not clear.

I do not expect you to do both parts. It is your own choice, and you are welcome to contact me if
you want to talk about which part to take.

Download loans.csv from Teams and save it in your data folder. Work in a new notebook. Always
keep the original: load it into loans_raw, and clean a copy that you call loans. Write a short
comment above each step, in your own words. If you get stuck: read the error message from the
bottom up, print the thing you are working with, and look at the lecture notebook. Then ask.

## Part 1: Practise

Clean the new table with the same steps as today. Same kinds of problems, new places.

Tools you need

loans_raw = pd.read_csv('data/loans.csv') loans = loans_raw.copy()
loans['x'].nunique() loans['x'].value_counts()
loans['x'].str.strip().str.lower() .str.title()
loans['x'].replace({'old': 'new'}) # whole values
loans['x'].str.replace('kr', '') # part of the text
loans['x'].map({'yes': True, 'no': False}) # unknown values become NaN
loans['x'].astype(float)
pd.to_datetime(loans['x'], errors='coerce') loans['x'].dt.to_period('M')
loans.duplicated().sum() loans.duplicated(subset='x').sum()
loans.drop_duplicates().reset_index(drop=True) loans.drop(columns='x')
loans.to_csv('data/loans_clean.csv', index=False)

## Tasks
## 1 Load and look

Load the file into loans_raw and make the copy loans. Print the shape and run info(). 
Write down in a comment every problem you can see already: which columns are text that
should be something else, and which column is empty?

In [2]:
import pandas as pd

loans_raw = pd.read_csv('data/loans.csv')
loans = loans_raw.copy()

print(" --- Shape of the dataset (Rows, Columns) ---\n")
print(loans.shape)

print("\n--- Dataset Info ---")
loans.info()

 --- Shape of the dataset (Rows, Columns) ---

(623, 9)

--- Dataset Info ---
<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   loan_id      623 non-null    str    
 1   member_id    623 non-null    str    
 2   title        623 non-null    str    
 3   genre        623 non-null    str    
 4   loan_date    623 non-null    str    
 5   return_date  593 non-null    str    
 6   late_fee     623 non-null    str    
 7   returned     623 non-null    str    
 8   notes        0 non-null      float64
dtypes: float64(1), str(8)
memory usage: 43.9 KB


## 2 Clean the genre

How many different values does genre have? It should be six genres. 
Remove spaces and capital letters with str.strip() and str.lower() and count again. 
Look at the values that are left with unique() and fix the rest with replace. 
Finish with str.title(). How many genres do you have, and how many loans in each?

In [3]:
# 2. Clean the genre column

# Check initial unique values count
print("Initial unique genres count:", loans['genre'].nunique())

# Strip whitespace and convert to lowercase
loans['genre'] = loans['genre'].str.strip().str.lower()

# Check values left or counts before fixing anomalies
print("\nUnique values after strip and lower:")
print(loans['genre'].value_counts(dropna=False))

# Fix remaining typos/anomalies using replace
loans['genre'] = loans['genre'].replace({
    'fictin': 'fiction',
    'fictioon': 'fiction',
    'scifi': 'sci-fi',
    'sci fi': 'sci-fi'
})

# Format nicely with str.title()
loans['genre'] = loans['genre'].str.title()

# Final check - number of unique genres and loan counts per genre
final_unique_count = loans['genre'].nunique()
genre_counts = loans['genre'].value_counts()

print(f"\nFinal number of unique genres: {final_unique_count}")
print("\n--- Number of loans in each genre ---")
print(genre_counts)

Initial unique genres count: 28

Unique values after strip and lower:
genre
crime              163
fantasy             90
romance             86
children            76
children's          59
history             59
science fiction     33
sci-fi              30
sci fi              27
Name: count, dtype: int64

Final number of unique genres: 8

--- Number of loans in each genre ---
genre
Crime              163
Fantasy             90
Romance             86
Children            76
Children'S          59
History             59
Sci-Fi              57
Science Fiction     33
Name: count, dtype: int64


## 3 Yes and no

The column returned says whether the book is back. How many different values does it have? 
Turn it into True and False with strip, lower and map. 
Check that nothing became missing, and count True and False.

In [4]:
## 3. Yes and no

# Check initial unique values in the column
print("Initial unique values in 'returned':")
print(loans['returned'].value_counts(dropna=False))

# Clean text by stripping spaces and converting to lowercase
cleaned_returned_text = loans['returned'].str.strip().str.lower()

# Map text values to True and False
loans['returned'] = cleaned_returned_text.map({
    'yes': True,
    'true': True,
    'y': True,
    'no': False,
    'false': False,
    'n': False
})

# Check if any unexpected values became missing (NaN)
missing_after_mapping = loans['returned'].isna().sum()
print(f"\nNumber of missing values (NaN) after mapping: {missing_after_mapping}")

# Count True and False values
print("\n--- Count of True and False values ---")
print(loans['returned'].value_counts(dropna=False))

Initial unique values in 'returned':
returned
Y      131
Yes    125
YES    124
y      113
yes    100
N        8
NO       7
No       6
n        6
no       3
Name: count, dtype: int64

Number of missing values (NaN) after mapping: 0

--- Count of True and False values ---
returned
True     593
False     30
Name: count, dtype: int64


## 4 The late fee is text

The late fee looks like 7,50 kr. Turn it into a number. First try astype(float) directly and read the error. 
Then remove kr, remove the spaces, change the comma to a dot, and only then use astype(float). 
Check the result with describe() and calculate the total of all fees.
On purpose: Try astype(float) on the original text first, and read the error from the bottom up.

In [7]:
## 4 The late fee is text

# On purpose: Trying to convert directly using astype(float) to see the error
# loans['late_fee'].astype(float)

# Clean the text by removing 'kr', stripping spaces, and replacing comma with dot
cleaned_fee = (
    loans['late_fee']
    .astype(str)                         # Check if everything is treated as string
    .str.replace('kr', '', regex=False)  # Remove 'kr'
    .str.strip()                         # Remove leading/trailing spaces
    .str.replace(',', '.', regex=False)  # Replace comma with dot for decimal notation
)

# Convert the cleaned text to float
loans['late_fee'] = cleaned_fee.astype(float)

# Check the result with describe()
print("--- Descriptive statistics for late_fee ---\n")
print(loans['late_fee'].describe().round(2))

# Calculate the total sum of all late fees
total_fees = loans['late_fee'].sum()
print(f"\nTotal sum of all late fees: {total_fees:.2f}")

--- Descriptive statistics for late_fee ---

count    623.00
mean      12.32
std       16.11
min        0.00
25%        0.00
50%        0.00
75%       25.00
max       47.50
Name: late_fee, dtype: float64

Total sum of all late fees: 7677.50


## 5 Dates
Turn loan_date into real dates with pd.to_datetime(..., errors='coerce'). 
How many dates became NaT? Find the first and the last date. 
Then count the loans per month with dt.to_period('M'), and check if the numbers add up to the whole table.

In [8]:
# 5. Dates

# Convert loan_date to datetime, forcing bad formats to NaT (Not a Time)
loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

# Count how many dates became NaT (invalid/unrecognized formats)
nat_count = loans['loan_date'].isna().sum()
print(f"Number of invalid dates (NaT): {nat_count}")

# Find the first (earliest) and last (latest) date in the column (min/max calculation)
earliest_date = loans['loan_date'].min()
latest_date = loans['loan_date'].max()

print(f"First loan date: {earliest_date}")
print(f"Last loan date: {latest_date}")

# Count loans per month using dt.to_period('M') and check sum
loans_per_month = loans['loan_date'].dt.to_period('M').value_counts().sort_index()

print("\n--- Loans per month ---\n")
print(loans_per_month)

# Check if the sum of monthly loans plus NaT equals the total rows in the table
total_monthly_loans = loans_per_month.sum()
total_table_rows = loans.shape[0]

print(f"\nSum of loans across all months: {total_monthly_loans}")
print(f"Total rows in the table: {total_table_rows}")
print(f"Do they match (accounting for NaT)? {total_monthly_loans + nat_count == total_table_rows}")

Number of invalid dates (NaT): 3
First loan date: 2026-01-05 00:00:00
Last loan date: 2026-06-27 00:00:00

--- Loans per month ---

loan_date
2026-01     96
2026-02    100
2026-03    111
2026-04    106
2026-05    108
2026-06     99
Freq: M, Name: count, dtype: int64

Sum of loans across all months: 620
Total rows in the table: 623
Do they match (accounting for NaT)? True


## 6 Duplicates
How many duplicates are there in the raw table, and how many after the cleaning? 
Why is the number different? Check with subset='loan_id'. 
Remove the duplicates, and reset the index. 
How much did the duplicates add to the total of the late fees, in kr and in percent?

In [9]:
# 6. Duplicates

# Check duplicates in the raw table
print("--- Duplicates in raw table ---")
print("Full row duplicates:", loans_raw.duplicated().sum())
print("Duplicates by loan_id:", loans_raw.duplicated(subset='loan_id').sum())

# Check duplicates in the cleaned table (before dropping)
print("\n--- Duplicates in cleaned table (before removal) ---")
print("Full row duplicates:", loans.duplicated().sum())
print("Duplicates by loan_id:", loans.duplicated(subset='loan_id').sum())

# Calculate how much duplicates added to the total late fees
total_fee_with_dups = loans['late_fee'].sum()
total_fee_without_dups = loans.drop_duplicates(subset='loan_id')['late_fee'].sum()

fee_diff_kr = total_fee_with_dups - total_fee_without_dups
fee_diff_percent = (fee_diff_kr / total_fee_without_dups) * 100

print(f"\n--- Late fee impact of duplicates ---")
print(f"Added amount: {fee_diff_kr:.2f} kr ({fee_diff_percent:.2f}%)")

# Remove duplicates based on loan_id and reset the index
loans = loans.drop_duplicates(subset='loan_id').reset_index(drop=True)

print(f"\nNew shape of loans table after removing duplicates: {loans.shape}")

--- Duplicates in raw table ---
Full row duplicates: 12
Duplicates by loan_id: 18

--- Duplicates in cleaned table (before removal) ---
Full row duplicates: 16
Duplicates by loan_id: 18

--- Late fee impact of duplicates ---
Added amount: 230.00 kr (3.09%)

New shape of loans table after removing duplicates: (605, 9)


## 7 Put it together
Remove the empty column notes. Then collect all your steps in one cell that starts from loans_raw and gives the finished table. 
Run it, check the shape, and save the table as data/loans_clean.csv. Run the cell twice. Does it work both times? Why?

In [10]:
# 7. Put it together: 

import pandas as pd

# Load raw data and make a copy
loans_raw = pd.read_csv('data/loans.csv')
loans = loans_raw.copy()

# Drop the empty column 'notes' (if it exists)
if 'notes' in loans.columns:
    loans = loans.drop(columns='notes')

# Clean 'genre' (strip spaces, lowercase, fix typos, title case)
loans['genre'] = (
    loans['genre']
    .str.strip()
    .str.lower()
    .replace({
        'fictin': 'fiction',
        'fictioon': 'fiction',
        'scifi': 'sci-fi',
        'sci fi': 'sci-fi'
    })
    .str.title()
)

# Clean 'returned' column and map to boolean
cleaned_returned = loans['returned'].str.strip().str.lower()
loans['returned'] = cleaned_returned.map({
    'yes': True,
    'true': True,
    'y': True,
    'no': False,
    'false': False,
    'n': False
})

# Clean 'late_fee' column and convert to float
cleaned_fee = (
    loans['late_fee']
    .astype(str)
    .str.replace('kr', '', regex=False)
    .str.strip()
    .str.replace(',', '.', regex=False)
)
loans['late_fee'] = cleaned_fee.astype(float)

# Clean 'loan_date' and convert to datetime
loans['loan_date'] = pd.to_datetime(loans['loan_date'], errors='coerce')

# Remove duplicates based on 'loan_id' and reset index
loans = loans.drop_duplicates(subset='loan_id').reset_index(drop=True)

# Save the cleaned table to CSV
loans.to_csv('data/loans_clean.csv', index=False)

# Print final shape to verify
print("--- Final cleaned table shape ---\n")
print(loans.shape)

--- Final cleaned table shape ---

(605, 8)


## Expected
Task 1: (623, 9). The column notes is completely empty. loan_date, return_date and late_fee are
text, and returned is text. return_date has 593 values (30 are missing).

Task 2: 28 values at first, then 9 after strip and lower. After the fix and title(): 6 genres. Crime
163, Children 135, Fantasy 90, Sci-Fi 90, Romance 86, History 59.

Task 3: 10 different values at first. After the fix: True 593 and False 30, with 0 missing.

Task 4: ValueError: could not convert string to float: '7,50 kr'. After the fix: min 0.0, max
47.5, mean 12.32, and a total of 7677.5.

Task 5: 3 dates become NaT. First date 2026-01-05, last date 2026-06-27. Per month: 96, 100, 111,
106, 108 and 99. They add up to 620, which is 623 minus the 3 NaT.

Task 6: 12 duplicates in the raw table and 18 after the cleaning (6 more, which differ only in
spelling). subset='loan_id' also gives 18. After removing: (605, 9). The total fee goes from 7677.5
to 7447.5, so the duplicates added 230.0 kr, about 3.0 percent.

Task 7: (605, 8). The cell gives the same result the second time, since it starts from loans_raw every
time

## Part 2: Go deeper

Here you meet problems that need a decision, and you write your own function. There is one new
tool (dt.days), and it is in the box below.

Tools you need

(loans['b'] - loans['a']).dt.days # days between two dates
loans[loans['b'] < loans['a']] # rows where b is before a
loans.duplicated(subset=['member_id', 'title', 'loan_date'], keep=False)
def clean_loans(raw):
table = raw.copy()
... # your steps
return table

## Tasks
## 2.1 Missing and impossible return dates

Add return_date to your one-cell cleaning from Part 1 (turn it into dates with errors='coerce', 
before you remove the duplicates) and run the cell again. 
How many return dates are missing? Compare with the number of loans where returned is False.
What does that tell you about why they are missing?
Then find the loans where the return date is before the loan date. How many are there?
Look at them and decide what to do with them. Write your decision in a comment, and what it does to the numbers. 
Calculate the number of days each loan lasted with (return_date - loan_date).dt.days and see the minimum.

In [12]:
import pandas as pd

def clean_loans(raw):
    # Create a working copy
    table = raw.copy()
    
    # Remove empty column 'notes' if it exists
    if 'notes' in table.columns:
        table = table.drop(columns='notes')

    # Clean genre (strip, lower, replace typos, title case)
    table['genre'] = (
        table['genre']
        .str.strip()
        .str.lower()
        .replace({
            'fictin': 'fiction',
            'fictioon': 'fiction',
            'scifi': 'sci-fi',
            'sci fi': 'sci-fi'
        })
        .str.title()
    )

    # Clean 'returned' column into boolean values
    cleaned_returned = table['returned'].str.strip().str.lower()
    table['returned'] = cleaned_returned.map({
        'yes': True,
        'true': True,
        'y': True,
        'no': False,
        'false': False,
        'n': False
    })

    # Clean 'late_fee' into numeric (float)
    cleaned_fee = (
        table['late_fee']
        .astype(str)
        .str.replace('kr', '', regex=False)
        .str.strip()
        .str.replace(',', '.', regex=False)
    )
    table['late_fee'] = cleaned_fee.astype(float)

    # Convert dates (loan_date and return_date) BEFORE removing duplicates
    table['loan_date'] = pd.to_datetime(table['loan_date'], errors='coerce')
    table['return_date'] = pd.to_datetime(table['return_date'], errors='coerce')



    
    # --- PART 2: HANDLING IMPOSSIBLE DATES ---
    # Find rows where return_date is strictly before loan_date
    impossible_dates_mask = table['return_date'] < table['loan_date']
    
    # DECISION (Comment): 
    # A book cannot be returned before it was borrowed (this indicates a data entry error).
    # Decision: Convert these impossible return dates to NaT (Not a Time) because we cannot know 
    # the exact return date, while keeping the rows to preserve the loan records themselves.
    
    table.loc[impossible_dates_mask, 'return_date'] = pd.NaT

    #Remove duplicates based on 'loan_id' and reset index
    table = table.drop_duplicates(subset='loan_id').reset_index(drop=True)

    return table

# Load raw data and apply our cleaning function
loans_raw = pd.read_csv('data/loans.csv')
loans = clean_loans(loans_raw)

# --- ANALYSIS FOR THE TASKS ---

# 1. Count missing return dates (NaT) and compare with returned == False
missing_return_dates = loans['return_date'].isna().sum()
not_returned_count = (loans['returned'] == False).sum()

print(f"Number of missing return dates (NaT): {missing_return_dates}")
print(f"Number of books not yet returned (returned == False): {not_returned_count}")

# 2. Calculate loan duration in days using .dt.days
loan_duration_days = (loans['return_date'] - loans['loan_date']).dt.days
print("\n--- Loan duration statistics (in days) ---")
print(loan_duration_days.describe().round(2))

# Save cleaned table
loans.to_csv('data/loans_clean.csv', index=False)
print(f"\nFinal cleaned table shape: {loans.shape}")

Number of missing return dates (NaT): 36
Number of books not yet returned (returned == False): 30

--- Loan duration statistics (in days) ---
count    566.00
mean      22.28
std       10.71
min        4.00
25%       13.00
50%       22.00
75%       32.00
max       40.00
dtype: float64

Final cleaned table shape: (605, 8)


## 2.2 Is it the same loan?
Some loans look the same but have different loan_id: same member, same book, same loan date. 
Find them with duplicated(subset=['member_id', 'title', 'loan_date'], keep=False). 
How many rows are there, and what is different between the two rows in each pair?
Are they the same loan that was registered twice, or two real loans? You can argue both ways. 
Decide, write your reasons in a comment, and calculate what your decision does to the total of the late fees.

In [13]:
# Part 2 - Task 2: Find potential duplicate loans with the same member, title, and loan date

# Find rows where member_id, title, and loan_date are identical but loan_id differs
duplicate_loans_mask = loans_raw.duplicated(subset=['member_id', 'title', 'loan_date'], keep=False)
duplicate_rows = loans_raw[duplicate_loans_mask].sort_values(by=['member_id', 'title', 'loan_date'])

print(f"Number of rows involved in these duplicates: {duplicate_rows.shape[0]}")
print("\n--- Sample of duplicate pairs ---")
print(duplicate_rows[['loan_id', 'member_id', 'title', 'loan_date', 'return_date', 'late_fee']].head(10))

Number of rows involved in these duplicates: 46

--- Sample of duplicate pairs ---
     loan_id member_id                title   loan_date return_date  late_fee
70   L-30068     M0006      A Quiet Witness  2026-01-24  2026-03-05  47,50 kr
71   L-30068     M0006      A Quiet Witness  2026-01-24  2026-03-05  47,50 kr
230  L-30223     M0007       Forest Friends  2026-03-11  2026-04-02   2,50 kr
231  L-30223     M0007       Forest Friends  2026-03-11  2026-04-02   2,50 kr
590  L-30571     M0008       Cold Case 1987  2026-06-19  2026-06-26    0,00kr
591  L-30571     M0008       Cold Case 1987  2026-06-19  2026-06-26    0,00kr
311  L-30301     M0010        Salt and Iron  2026-04-03  2026-04-22    0,00kr
312  L-30301     M0010        Salt and Iron  2026-04-03  2026-04-22    0,00kr
361  L-30350     M0019  Letters from Lisbon  2026-04-16  2026-05-25  45,00 kr
362  L-30350     M0019  Letters from Lisbon  2026-04-16  2026-05-25  45,00 kr


## 3 Make it reusable
Write a function clean_loans(raw) that takes the raw table and returns the finished table,
with all the steps from Part 1 and Part 2 (the genre, returned, the late fee, loan_date, return_date, duplicates and notes). 
Test it on loans_raw. Check the shape, the total of the fees and the number of missing return dates. 
Why is a function better than a long list of cells?

## 4 Fees by genre
Use your table without the pairs from task 2 (or explain if you keep them). 
For every genre, calculate the number of loans, the share of loans with a late fee above 0, and the average late fee. 
Which genre has the highest share of late loans? Write what you can and cannot conclude.
Data analysis & visualisation · Lab 3 Page 4


## 5 Your own question
Come up with one question about the library data that you can answer with the tools from today. 
Write the question as a comment, answer it with code, and finish with a comment about what the answer does not tell you. 
That last part is the most important one.


## Expected
Task 1: 30 return dates are missing, and 30 loans have returned equal to False: the book is not
back yet, so there is no return date. That is missing for a reason. 6 loans have a return date before
the loan date, with a shortest length of -5 days. Mean length without them: 22.28 days.

Task 2: 10 rows, 5 pairs, e.g. member M0023 and "The Locksmith" on 2026-03-15. The loan_id is
the only difference (L-30238 and L-40004). If you remove one of each pair, the total fee goes from
7447.5 to 7362.5 (85.0 kr less) and the table has 600 rows.

Task 3: the function gives (605, 8), a total fee of 7447.5 and 30 missing return dates, the same as
the step by step result.

Task 4: Sci-Fi has the highest share of late loans (0.539, 89 loans) and Fantasy the lowest (0.448, 87
loans).
Stuck? Read the error message from the bottom up, print the data you are working with, and then ask.